- Folder name can be set according to your need
- TEST_PAT_SIZE can be assigned to either **"small"** or **"large"**
  - **"small"** stands for small test pattern size
  - **"large"** stands for large test pattern size
  - During implementation, it is recommended to set TEST_PAT_SIZE **"small"** for quick debugging. However, you have to pass the unit test with both TEST_PAT_SIZE **"small"** and **"large"** to get the full score in each part.
  - Note that for **"large"** pattern size, the bilateral filtering process may take longer time to complete.


In [ ]:
from pathlib import Path
import os


def find_hw01_root():
    cwd = Path.cwd().resolve()
    for path in (cwd, *cwd.parents):
        if (path / 'code' / 'HDR_functions.ipynb').exists() and (path / 'UnitTestPat_small').exists():
            return Path(os.path.relpath(path, cwd))

        hw01 = path / 'hw01'
        if (hw01 / 'code' / 'HDR_functions.ipynb').exists() and (hw01 / 'UnitTestPat_small').exists():
            return Path(os.path.relpath(hw01, cwd))

    raise FileNotFoundError('Run this notebook from the repository root, hw01, or hw01/code.')


FOLDER_NAME = find_hw01_root()

# TEST_PAT_SIZE = 'small'
TEST_PAT_SIZE = 'large'


Do not modify the remaining code.

In [ ]:
import unittest
import numpy as np
from functools import partial

In [ ]:
%run "{FOLDER_NAME}/code/HDR_functions.ipynb"

In [ ]:
''' Test functions in HDR flow '''
def Cal_PSNR(img1, img2):
    img1 = img1.astype(np.float32)
    img2 = img2.astype(np.float32)
    mse = np.mean((img1 - img2) ** 2)
    if mse < 6.5025e-6:
        psnr = 100  # set upper bound to avoid divided by zero
    else:
        psnr = 10 * np.log10((255.0 ** 2)/mse)
    return psnr

class Test_HDR_functions(unittest.TestCase):
    def test1_Response_Estimation(self):
        img_samples = np.load(f'{FOLDER_NAME}/UnitTestPat_{TEST_PAT_SIZE}/img_samples_1ch.npy')
        etime_list = np.load(f'{FOLDER_NAME}/UnitTestPat_{TEST_PAT_SIZE}/etime_list.npy')
        golden = np.load(f'{FOLDER_NAME}/UnitTestPat_{TEST_PAT_SIZE}/resp_1ch.npy')
        resp_test = Response_Estimation(img_samples, etime_list, lambda_=50)
        mse = np.mean((golden - resp_test)**2)
        self.assertLessEqual(mse, 0.01)
        return mse

    def test2_Radiance_Construction(self):
        img_list = np.load(f'{FOLDER_NAME}/UnitTestPat_{TEST_PAT_SIZE}/img_list_1ch.npy')
        resp = np.load(f'{FOLDER_NAME}/UnitTestPat_{TEST_PAT_SIZE}/resp_1ch.npy')
        etime_list = np.load(f'{FOLDER_NAME}/UnitTestPat_{TEST_PAT_SIZE}/etime_list.npy')
        golden = np.load(f'{FOLDER_NAME}/UnitTestPat_{TEST_PAT_SIZE}/rad_1ch.npy')
        rad_test = Radiance_Construction(img_list, resp, etime_list)
        mse = np.mean((golden - rad_test)**2)
        self.assertLessEqual(mse, 0.01)
        return mse

    def test3_White_Balance_Adjustment(self):
        src = np.load(f'{FOLDER_NAME}/UnitTestPat_{TEST_PAT_SIZE}/rad.npy')
        y_range = np.load(f'{FOLDER_NAME}/UnitTestPat_{TEST_PAT_SIZE}/y_range.npy')
        x_range = np.load(f'{FOLDER_NAME}/UnitTestPat_{TEST_PAT_SIZE}/x_range.npy')
        golden = np.load(f'{FOLDER_NAME}/UnitTestPat_{TEST_PAT_SIZE}/rad_wb.npy')
        wb_test = White_Balance(src, y_range, x_range)
        mse = np.mean((golden - wb_test)**2)
        self.assertLessEqual(mse, 0.01)
        return mse

    def test4_Global_Tone_Mapping(self):
        src = np.load(f'{FOLDER_NAME}/UnitTestPat_{TEST_PAT_SIZE}/rad_wb.npy')
        golden = ReadImg(f'{FOLDER_NAME}/UnitTestPat_{TEST_PAT_SIZE}/memorial_gtm.png')
        gtm_test = Global_Tone_Mapping(src, scale=2.0)
        psnr = Cal_PSNR(golden, gtm_test)
        self.assertGreaterEqual(psnr, 45)
        return psnr

    def test5_Gaussian(self):
        src = np.load(f'{FOLDER_NAME}/UnitTestPat_{TEST_PAT_SIZE}/L.npy')
        golden = np.load(f'{FOLDER_NAME}/UnitTestPat_{TEST_PAT_SIZE}/filter_gau_golden.npy')
        gau_test = Gaussian_Filter(src, N=15, sigma_s=100)
        mse = np.mean((golden - gau_test)**2)
        self.assertLessEqual(mse, 0.01)
        return mse

    def test6_Local_Tone_Mapping_gaussian(self):
        src = np.load(f'{FOLDER_NAME}/UnitTestPat_{TEST_PAT_SIZE}/rad_wb.npy')
        golden = ReadImg(f'{FOLDER_NAME}/UnitTestPat_{TEST_PAT_SIZE}/memorial_ltm_gau.png')
        gau = partial(Gaussian_Filter, N=15, sigma_s=100)
        ltm_gau_test = Local_Tone_Mapping(src, gau, scale=7)
        psnr = Cal_PSNR(golden, ltm_gau_test)
        self.assertGreaterEqual(psnr, 45)
        return psnr

    def test7_Bilateral(self):
        src = np.load(f'{FOLDER_NAME}/UnitTestPat_{TEST_PAT_SIZE}/L.npy')
        golden = np.load(f'{FOLDER_NAME}/UnitTestPat_{TEST_PAT_SIZE}/filter_bil_golden.npy')
        bil_test = Bilateral_Filter(src, N=15, sigma_s=100, sigma_r=0.8)
        mse = np.mean((golden - bil_test)**2)
        self.assertLessEqual(mse, 0.01)
        return mse

    def test8_Local_Tone_Mapping_bilateral(self):
        src = np.load(f'{FOLDER_NAME}/UnitTestPat_{TEST_PAT_SIZE}/rad_wb.npy')
        golden = ReadImg(f'{FOLDER_NAME}/UnitTestPat_{TEST_PAT_SIZE}/memorial_ltm_bil.png')
        bil = partial(Bilateral_Filter, N=15, sigma_s=100, sigma_r=0.8)
        ltm_bil_test = Local_Tone_Mapping(src, bil, scale=7)
        psnr = Cal_PSNR(golden, ltm_bil_test)
        self.assertGreaterEqual(psnr, 45)
        return psnr


In [ ]:
testcases = unittest.TestLoader().loadTestsFromTestCase(Test_HDR_functions)
unittest.TextTestRunner(verbosity=2).run(testcases)